In [16]:
from datasets import load_dataset  


ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset" , split="train") 
ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset" , split="train") 
ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset" , split="train")
ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train") 



In [17]:
ds1.num_rows , ds2.num_rows , ds3.num_rows , ds4.num_rows

(71179, 100000, 100000, 500)

In [28]:
## we randomly combined the dataset so model is able to understand most randomness.  
import logging
import random  
from datasets import concatenate_datasets
log = logging.getLogger(__name__)
SEED = 42 

WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
}
TOTAL_ROWS = 200000
def weighted_combine(ds1 , ds2 , ds3 , ds4):
    random.seed(SEED) 
    source_map = {
        "instruction" : ds1 , 
        "summarization":ds2 , 
        "extraction" : ds3 , 
        "decomposition" : ds4
    } 
    selected_split = [] 
    for name , ds in source_map.items(): 
        target_n = int(TOTAL_ROWS*WEIGHTS[name]) 
        if len(ds)>=target_n: 
            idx = random.sample(range(len(ds)) , target_n) 
            split = ds.select(idx) 
        else: 
            # Upsample with repetition (important for small D4)
            repeats = (target_n // len(ds)) + 1
            idx = (list(range(len(ds))) * repeats)[:target_n]
            random.shuffle(idx)
            split = ds.select(idx)
            log.info(f"  ↑ Upsampled {name}: {len(ds):,} → {target_n:,}") 
        selected_split.append(split) 
        log.info(f"  {name:<15} → {target_n:,} samples ({WEIGHTS[name]*100:.0f}%)")
    
    combined = concatenate_datasets(selected_split)
    print(combined)
    # Shuffle the final combined dataset
    combined = combined.shuffle(seed=SEED)
    log.info(f"  ✅ Combined total: {len(combined):,} samples")
    return combined


In [29]:
final_ds = weighted_combine(ds1,ds2,ds3,ds4)

Dataset({
    features: ['dataset', 'task', 'system', 'messages'],
    num_rows: 200000
})
